# Phase 2: verify pipeline.search() live, tune tau_high/tau_low

Three Phase 2 checklist items in one run, since they all need the same live
`pipeline.search()` calls against real Qdrant + real models:

1. **Package retrieval as a module, models loaded once** -- verify a second
   query returns under 1s and memory is flat across 100 requests.
2. **Make the English leg conditional on the Sindhi top score** -- compare
   median latency and Recall@1 with the leg forced off vs conditional.
3. **Tune tau_high and tau_low** -- from the 275-query corrected gold set
   (correct/incorrect top-1 scores) and the 100-query negative set (scores
   for queries with no correct KB answer).

**Runtime -> Change runtime type -> T4 GPU.**

In [ ]:
!pip install -q qdrant-client FlagEmbedding transformers sentencepiece psutil matplotlib

In [ ]:
%cd /content
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

print("cloned OK")

In [ ]:
import os
from getpass import getpass

# pipeline.py reads these from os.environ directly (12-factor style) --
# set them as real env vars, not just local Python variables.
os.environ["QDRANT_URL"] = getpass("Qdrant cluster URL: ")
os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")

print("env vars set")

In [ ]:
import csv

with open("eval/gold_eval_280_linked.csv", encoding="utf-8-sig", newline="") as f:
    gold_rows = list(csv.DictReader(f))

with open("eval/negative_set_100.csv", encoding="utf-8-sig", newline="") as f:
    negative_rows = list(csv.DictReader(f))

print(f"{len(gold_rows)} gold rows (expect 275), {len(negative_rows)} negative rows (expect 100)")
assert len(gold_rows) == 275
assert len(negative_rows) == 100

## Item 1: warm-loading, latency, memory

First call pays every model's load cost (bge-m3, bge-reranker-v2-m3, and
NLLB *only if* the first query happens to be uncertain enough to trigger the
English leg). Second call, and every call after, should be warm.

In [ ]:
import time

from retrieval.pipeline import search, warmup

# Force every model to load now (embed, rerank, AND translate) instead of
# letting translate's NLLB load lazily whenever the first uncertain query
# happens to show up -- that was the bug the first run of this notebook
# hit: query 1 didn't need the English leg, so query 2 silently paid
# NLLB's ~60s cold-load cost and failed the "under 1s" check below.
t0 = time.perf_counter()
warmup()
warmup_ms = (time.perf_counter() - t0) * 1000
print(f"warmup(): {warmup_ms:.0f}ms (loads bge-m3, bge-reranker-v2-m3, NLLB, and connects to Qdrant)")

t0 = time.perf_counter()
first = search(gold_rows[0]["query"])
first_ms = (time.perf_counter() - t0) * 1000
print(f"first real query (post-warmup): {first_ms:.0f}ms -- reported latency_ms: {first['latency_ms']}")

t0 = time.perf_counter()
second = search(gold_rows[1]["query"])
warm_ms = (time.perf_counter() - t0) * 1000
print(f"second real query: {warm_ms:.0f}ms -- reported latency_ms: {second['latency_ms']}")
assert warm_ms < 1000, f"second call took {warm_ms:.0f}ms, expected under 1000ms"

In [ ]:
import psutil

process = psutil.Process(os.getpid())
mem_samples = []

for i, row in enumerate(gold_rows[:100], start=1):
    search(row["query"])
    if i % 10 == 0:
        rss_mb = process.memory_info().rss / (1024 ** 2)
        mem_samples.append((i, rss_mb))
        print(f"{i}/100 -- RSS: {rss_mb:.0f}MB")

first_half = [m for i, m in mem_samples if i <= 50]
second_half = [m for i, m in mem_samples if i > 50]
growth_mb = (sum(second_half) / len(second_half)) - (sum(first_half) / len(first_half))
print(f"\nmean RSS first 50 calls: {sum(first_half)/len(first_half):.0f}MB, "
      f"last 50: {sum(second_half)/len(second_half):.0f}MB, growth: {growth_mb:+.0f}MB")
print("flat (no leak) if growth is small relative to baseline RSS, not a per-call increment.")

## Item 3 setup: collect top-1 scores for tau tuning

Runs every gold query and every negative-set query through the real
pipeline once (default `tau_high` from the `TAU_HIGH` env var / 0.75
fallback), recording the top-1 score and, for gold queries, whether it was
correct.

In [ ]:
gold_scored = []
for i, row in enumerate(gold_rows, start=1):
    result = search(row["query"], top_k=1)
    top1 = result["results"][0] if result["results"] else None
    gold_scored.append({
        "query_id": row["query_id"],
        "correct_answer_id": row["correct_answer_id"],
        "top1_id": top1["answer_id"] if top1 else None,
        "score": top1["score"] if top1 else 0.0,
        "correct": bool(top1) and str(top1["answer_id"]) == str(row["correct_answer_id"]),
    })
    if i % 40 == 0:
        print(f"gold {i}/{len(gold_rows)}")

print(f"done, {len(gold_scored)} gold queries scored")

In [ ]:
negative_scored = []
for i, row in enumerate(negative_rows, start=1):
    result = search(row["question"], top_k=1)
    top1 = result["results"][0] if result["results"] else None
    negative_scored.append({
        "id": row["id"],
        "score": top1["score"] if top1 else 0.0,
    })
    if i % 25 == 0:
        print(f"negative {i}/{len(negative_rows)}")

print(f"done, {len(negative_scored)} negative queries scored")

## Item 3: pick tau_high and tau_low

tau_high = smallest score threshold where precision("top-1 is correct") on
the gold set reaches >=0.95. tau_low = the score below which >=90% of the
negative set's own top-1 scores fall, per docs/PLAYBOOKS.md Lever 5.

In [ ]:
sorted_by_score = sorted(gold_scored, key=lambda r: -r["score"])

tau_high = None
for cutoff_row in sorted_by_score:
    threshold = cutoff_row["score"]
    at_or_above = [r for r in gold_scored if r["score"] >= threshold]
    precision = sum(r["correct"] for r in at_or_above) / len(at_or_above)
    if precision >= 0.95:
        tau_high = threshold
        coverage = len(at_or_above) / len(gold_scored)

print(f"tau_high = {tau_high:.4f}" if tau_high else "no threshold reaches 0.95 precision anywhere")
if tau_high:
    print(f"at this threshold: precision={precision:.3f}, coverage={coverage:.3f} "
          f"({int(coverage*len(gold_scored))}/{len(gold_scored)} gold queries clear it)")

In [ ]:
negative_scores_sorted = sorted(r["score"] for r in negative_scored)
p90_index = int(0.90 * len(negative_scores_sorted))
tau_low = negative_scores_sorted[p90_index]

below = sum(1 for s in negative_scores_sorted if s < tau_low)
print(f"tau_low = {tau_low:.4f} -- {below}/{len(negative_scores_sorted)} "
      f"({below/len(negative_scores_sorted):.1%}) of the negative set falls below it")

if tau_high and tau_low >= tau_high:
    print("\nWARNING: tau_low >= tau_high -- the two bands overlap or invert. "
          "This means correct and negative-set scores aren't cleanly separated; "
          "don't ship these values without a closer look.")

In [ ]:
import matplotlib.pyplot as plt

correct_scores = [r["score"] for r in gold_scored if r["correct"]]
incorrect_scores = [r["score"] for r in gold_scored if not r["correct"]]
neg_scores = [r["score"] for r in negative_scored]

fig, ax = plt.subplots(figsize=(9, 5))
bins = 30
ax.hist(correct_scores, bins=bins, alpha=0.6, label=f"gold: correct top-1 (n={len(correct_scores)})", color="#2a9d8f")
ax.hist(incorrect_scores, bins=bins, alpha=0.6, label=f"gold: incorrect top-1 (n={len(incorrect_scores)})", color="#e76f51")
ax.hist(neg_scores, bins=bins, alpha=0.6, label=f"negative set (n={len(neg_scores)})", color="#264653")
if tau_high:
    ax.axvline(tau_high, color="#2a9d8f", linestyle="--", label=f"tau_high = {tau_high:.3f}")
ax.axvline(tau_low, color="#264653", linestyle="--", label=f"tau_low = {tau_low:.3f}")
ax.set_xlabel("pipeline.search() top-1 score")
ax.set_ylabel("count")
ax.set_title("Score distributions: correct vs incorrect (gold) vs negative set")
ax.legend()
fig.tight_layout()
fig.savefig("eval/tau_score_distributions.png", dpi=150)
plt.show()
print("saved eval/tau_score_distributions.png")

## Item 2: conditional English leg -- does it actually save latency without costing recall?

Re-runs the gold set twice: once with the English leg forced off
(`tau_high=0.0`, so the Sindhi-only score always already "clears" the bar),
once with the tuned `tau_high` from above (conditional -- only translates
when genuinely uncertain).

In [ ]:
import statistics

def run_pass(tau):
    latencies, correct = [], 0
    for row in gold_rows:
        t0 = time.perf_counter()
        result = search(row["query"], top_k=1, tau_high=tau)
        latencies.append((time.perf_counter() - t0) * 1000)
        top1 = result["results"][0] if result["results"] else None
        if top1 and str(top1["answer_id"]) == str(row["correct_answer_id"]):
            correct += 1
    return latencies, correct / len(gold_rows)

sd_only_latencies, sd_only_recall = run_pass(tau=0.0)
print(f"English leg forced OFF -- median latency: {statistics.median(sd_only_latencies):.0f}ms, "
      f"Recall@1: {sd_only_recall:.3f}")

conditional_latencies, conditional_recall = run_pass(tau=tau_high or 0.75)
print(f"conditional (tau_high={tau_high or 0.75:.3f}) -- median latency: "
      f"{statistics.median(conditional_latencies):.0f}ms, Recall@1: {conditional_recall:.3f}")

In [ ]:
import datetime

lines = []
lines.append("\n\n# Phase 2 -- pipeline verification and tau tuning\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z, via retrieval/scripts/verify_pipeline_and_tune_thresholds.ipynb\n\n")

lines.append("## Item 1 -- warm loading, latency, memory\n")
lines.append(f"Explicit `warmup()` (loads bge-m3, bge-reranker-v2-m3, NLLB, connects to Qdrant): {warmup_ms:.0f}ms. "
              f"First real query post-warmup: {first_ms:.0f}ms. Second real query: {warm_ms:.0f}ms "
              f"(target: under 1000ms). Memory across 100 calls: first-50 mean {sum(first_half)/len(first_half):.0f}MB, "
              f"last-50 mean {sum(second_half)/len(second_half):.0f}MB, growth {growth_mb:+.0f}MB.\n\n")

lines.append("## Item 2 -- conditional English leg\n")
lines.append("| Mode | Median latency | Recall@1 |\n|---|---:|---:|\n")
lines.append(f"| English leg forced off | {statistics.median(sd_only_latencies):.0f}ms | {sd_only_recall:.3f} |\n")
lines.append(f"| Conditional (tau_high={tau_high or 0.75:.3f}) | {statistics.median(conditional_latencies):.0f}ms | {conditional_recall:.3f} |\n\n")

lines.append("## Item 3 -- tau_high / tau_low\n")
lines.append(f"**tau_high = {tau_high:.4f}**" if tau_high else "**tau_high: no threshold reached 0.95 precision**")
if tau_high:
    lines.append(f" -- precision {precision:.3f}, coverage {coverage:.3f} "
                  f"({int(coverage*len(gold_scored))}/{len(gold_scored)} gold queries)\n\n")
else:
    lines.append("\n\n")
lines.append(f"**tau_low = {tau_low:.4f}** -- {below}/{len(negative_scores_sorted)} "
              f"({below/len(negative_scores_sorted):.1%}) of the negative set falls below it (target >=90%).\n\n")
lines.append("Score distribution figure: `eval/tau_score_distributions.png`.\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

In [ ]:
from google.colab import files
files.download("eval/results.md")
files.download("eval/tau_score_distributions.png")